# PropNavigator - Model Error Analysis

**Diagnostic notebook - NOT part of the training pipeline.** Re-run it whenever the model is retrained.

It loads the deployed model (`artifacts/best_model.joblib`) and asks: *where is the model accurate, and where does it struggle?* - broken down by price band, property type, and sector.

It reproduces the **exact** train/test split used in training (stratified on price quintiles, `test_size=0.2`, `random_state=42`), so the numbers here match the model's real held-out performance.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score, mean_absolute_error,
    mean_absolute_percentage_error, median_absolute_error,
)

pd.set_option('display.max_rows', 120)
plt.rcParams['figure.figsize'] = (8, 4)

# Find the project root from wherever this notebook is opened.
ROOT = Path.cwd()
while not (ROOT / 'artifacts').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print('Project root:', ROOT)

## 1. Load data and reproduce the exact test split

In [ ]:
TARGET_COL = 'price_in_cr'

df = pd.read_csv(ROOT / 'data/fs/feature_selected_properties.csv')
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]
y_log = np.log1p(y)                          # model is trained on log(price)
price_bins = pd.qcut(y, q=5, labels=False)   # same stratification as training

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, stratify=price_bins, test_size=0.2, random_state=42
)
print('Test set:', X_test.shape)

## 2. Load the deployed model and predict

In [ ]:
bundle = joblib.load(ROOT / 'artifacts/best_model.joblib')
pipeline = bundle['pipeline']
print('Deployed model:', bundle['model_name'], '| saved MAPE:', bundle['test_mape_percent'], '%')

# Predict on the held-out test set, back on the original price scale (Cr).
y_pred = np.expm1(pipeline.predict(X_test))
y_true = np.expm1(y_test_log).values
residuals = y_true - y_pred
abs_errors = np.abs(residuals)
pct_errors = np.abs(residuals / y_true) * 100

## 3. Overall metrics

In [ ]:
summary = pd.DataFrame([
    ('MAPE (%)',       round(mean_absolute_percentage_error(y_true, y_pred) * 100, 2)),
    ('MdAPE (%)',      round(float(np.median(pct_errors)), 2)),
    ('MAE (Cr)',       round(mean_absolute_error(y_true, y_pred), 3)),
    ('R2',             round(r2_score(y_true, y_pred), 4)),
    ('P90 error (%)',  round(float(np.percentile(pct_errors, 90)), 2)),
    ('P95 error (%)',  round(float(np.percentile(pct_errors, 95)), 2)),
    ('Max error (%)',  round(float(pct_errors.max()), 2)),
    ('Test count',     int(len(y_true))),
], columns=['metric', 'value'])
summary

## 4. Where is the model weak? Segment breakdown

MAPE / MAE / R2 computed per segment. Segments with fewer than 5 rows are skipped (too few to be meaningful).

In [ ]:
def segment_metrics(y_true, y_pred, labels, name):
    labels = pd.Series(labels).reset_index(drop=True)
    yt = pd.Series(y_true).reset_index(drop=True)
    yp = pd.Series(y_pred).reset_index(drop=True)
    valid = labels.dropna()
    if isinstance(valid.dtype, pd.CategoricalDtype) and valid.cat.ordered:
        segs = [c for c in valid.cat.categories if (valid == c).any()]
    else:
        segs = sorted(valid.unique())
    rows = []
    for s in segs:
        m = labels == s
        if m.sum() < 5:
            continue
        rows.append({
            'segment': s, 'segment_type': name, 'count': int(m.sum()),
            'mape_pct': round(mean_absolute_percentage_error(yt[m], yp[m]) * 100, 2),
            'mae_cr': round(mean_absolute_error(yt[m], yp[m]), 3),
            'median_ae_cr': round(median_absolute_error(yt[m], yp[m]), 3),
            'r2': round(r2_score(yt[m], yp[m]), 4),
        })
    return pd.DataFrame(rows)

### 4a. By price bracket

In [ ]:
price_brackets = pd.cut(
    pd.Series(y_true), bins=[0, 1, 3, 5, 10, np.inf],
    labels=['<1 Cr', '1-3 Cr', '3-5 Cr', '5-10 Cr', '10+ Cr']
)
seg_price = segment_metrics(y_true, y_pred, price_brackets, 'price_bracket')
seg_price

### 4b. By property type

In [ ]:
seg_ptype = segment_metrics(y_true, y_pred, X_test['property_type'].values, 'property_type')
seg_ptype

### 4c. By sector (worst and best 10 by MAPE)

In [ ]:
seg_sector = segment_metrics(y_true, y_pred, X_test['sector'].values, 'sector')
print('Worst 10 sectors by MAPE:')
display(seg_sector.sort_values('mape_pct', ascending=False).head(10))
print('\nBest 10 sectors by MAPE:')
display(seg_sector.sort_values('mape_pct').head(10))

## 5. The 30 worst predictions (largest absolute error)

In [ ]:
worst = X_test.copy()
worst['y_true_cr'] = np.round(y_true, 3)
worst['y_pred_cr'] = np.round(y_pred, 3)
worst['abs_error_cr'] = np.round(abs_errors, 3)
worst['pct_error'] = np.round(pct_errors, 2)
cols = ['property_type', 'sector', 'area', 'bedRoom', 'y_true_cr', 'y_pred_cr', 'abs_error_cr', 'pct_error']
worst.sort_values('abs_error_cr', ascending=False).head(30)[cols]

## 6. Visual diagnostics

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

# Predicted vs actual (clipped at 99th pct so luxury outliers don't squash the view)
lim = [0, float(np.percentile(y_true, 99))]
ax[0].scatter(y_true, y_pred, s=6, alpha=0.3)
ax[0].plot(lim, lim, 'r--')
ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel('Actual (Cr)'); ax[0].set_ylabel('Predicted (Cr)')
ax[0].set_title('Predicted vs Actual')

# MAPE by price bracket
ax[1].bar(seg_price['segment'].astype(str), seg_price['mape_pct'])
ax[1].set_ylabel('MAPE %'); ax[1].set_title('MAPE by price bracket')
ax[1].tick_params(axis='x', rotation=45)

# Error distribution (clip extreme tail for readability)
ax[2].hist(pct_errors[pct_errors < 100], bins=50)
ax[2].set_xlabel('Absolute % error'); ax[2].set_title('Error distribution')

plt.tight_layout(); plt.show()

## 7. Save the breakdown (optional)

Writes the summary, per-segment metrics, and worst predictions to `data/error_analysis/` for reference.

In [ ]:
out = ROOT / 'data/error_analysis'
out.mkdir(parents=True, exist_ok=True)
summary.to_csv(out / 'error_summary.csv', index=False)
pd.concat([seg_price, seg_ptype, seg_sector], ignore_index=True).to_csv(out / 'segment_metrics.csv', index=False)
worst.sort_values('abs_error_cr', ascending=False).head(30)[cols].to_csv(out / 'worst_predictions.csv', index=False)
print('Saved to', out)

## 8. Reading the results (interview notes)

Typical pattern for a real-estate price model:

- **Percentage error usually grows at the extremes** - very cheap (<1 Cr) and luxury (10+ Cr) listings are harder than the dense 1-5 Cr middle, because there are fewer comparable sales to learn from.
- **Some sectors have high MAPE** because of thin data or genuine price heterogeneity (mixed builder quality on the same road).
- The **worst-prediction table** often surfaces data issues (mislabelled area, unusual listings) as much as model weakness.

Fill in your own observations here after running the notebook.